In [19]:


from scipy.stats import shapiro
import numpy as np
import pickle
import pandas as pd
from sklearn.preprocessing import QuantileTransformer


#importing data

try:
    with open('frameVectorAnnots.pkl', 'rb') as f:
        annots = pickle.load(f)
except:
    print("Error")

## checking normality of features
If majority are normal will proceed with normal standardization, otherwise will use feature rank normalization

In [20]:
#stacking all frames into matrix
frameList = [] 

frameIds = [] 

for day in annots:
    for operation in annots[day]:
        for frame in annots[day][operation]:
            frameList.append(frame['frameVector'])
            frameIds.append((day, operation, frame['frameIdx']))


frameMatrix = np.array(frameList)
print(np.shape(frameMatrix))

#------converting matrix into pandas data frame--

#creating cols and rows
dfCols = []
dfRows = []

interpersonIdx = 1
tableFeatIdx = 1
for i in range(1, np.shape(frameMatrix)[1]+1):
    if i <=50:
        dfCols.append(f"Gpd_{i}")

    elif i == 51:
        dfCols.append(f"Num_clinicians")
    
    elif i>=52 and i<=55:
        dfCols.append(f"Table_feature_{tableFeatIdx}")
        tableFeatIdx+=1
    
    else: 
        dfCols.append(f"Interperson_feature_{interpersonIdx}")
        interpersonIdx+=1

for i in range(1, np.shape(frameMatrix)[0]+1):
    dfRows.append(f"Frame_{i}")

(626, 64)


In [21]:
print(dfCols)

['Gpd_1', 'Gpd_2', 'Gpd_3', 'Gpd_4', 'Gpd_5', 'Gpd_6', 'Gpd_7', 'Gpd_8', 'Gpd_9', 'Gpd_10', 'Gpd_11', 'Gpd_12', 'Gpd_13', 'Gpd_14', 'Gpd_15', 'Gpd_16', 'Gpd_17', 'Gpd_18', 'Gpd_19', 'Gpd_20', 'Gpd_21', 'Gpd_22', 'Gpd_23', 'Gpd_24', 'Gpd_25', 'Gpd_26', 'Gpd_27', 'Gpd_28', 'Gpd_29', 'Gpd_30', 'Gpd_31', 'Gpd_32', 'Gpd_33', 'Gpd_34', 'Gpd_35', 'Gpd_36', 'Gpd_37', 'Gpd_38', 'Gpd_39', 'Gpd_40', 'Gpd_41', 'Gpd_42', 'Gpd_43', 'Gpd_44', 'Gpd_45', 'Gpd_46', 'Gpd_47', 'Gpd_48', 'Gpd_49', 'Gpd_50', 'Num_clinicians', 'Table_feature_1', 'Table_feature_2', 'Table_feature_3', 'Table_feature_4', 'Interperson_feature_1', 'Interperson_feature_2', 'Interperson_feature_3', 'Interperson_feature_4', 'Interperson_feature_5', 'Interperson_feature_6', 'Interperson_feature_7', 'Interperson_feature_8', 'Interperson_feature_9']


In [22]:
frameVectorDf = pd.DataFrame(
    frameMatrix,
    columns = dfCols,
    index = dfRows
)

frameVectorDf.head()

,Gpd_1,Gpd_2,Gpd_3,Gpd_4,Gpd_5,Gpd_6,Gpd_7,Gpd_8,Gpd_9,Gpd_10,...,Table_feature_4,Interperson_feature_1,Interperson_feature_2,Interperson_feature_3,Interperson_feature_4,Interperson_feature_5,Interperson_feature_6,Interperson_feature_7,Interperson_feature_8,Interperson_feature_9
Frame_1,9.522396,9.745949,-2.441483,-2.424845,0.847117,-3.612987,-7.905068,-4.840728,7.682698,2.057512,...,1767.585850,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_2,-6.240740,7.566539,4.392485,-3.616573,8.635847,-11.345625,-5.310072,4.286849,0.439230,3.040247,...,2622.616756,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_3,-7.432102,-9.433077,-2.599083,-2.519187,6.199976,1.432799,-7.219241,7.671942,0.163997,3.052592,...,2445.534774,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_4,-10.756118,-3.976860,-1.695456,-4.020653,6.824022,-1.232486,-3.858355,-1.802732,6.092485,-1.278292,...,2456.068758,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_5,-9.633286,-4.889347,0.764947,-1.182092,8.913971,3.765080,-4.183628,3.302828,1.759320,2.955471,...,2400.928740,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# computes shapiro wilk for each in feature in the data frame
    # Ho: data is normally distributed vs Ha: data is non normally distributed
# returns a dataframe showing which features are normally distributed and which arent
def computeShapiro(df, alpha = 0.05):
    results = []
    rows  = [] 
    cols = ["Normality"] 
    for colName, colData in df.items():
        rows.append(colName)

        #checking normality
        _, p = shapiro(colData)
        normality = 0 if p<=alpha else 1   #1 if data is normal, 0 if not
        results.append(normality)

    featureDistrib = pd.DataFrame(
        results,
        columns = cols,
        index = rows
    )

    #compute percentage of normally distributed features
    normalFeatures = len(featureDistrib.query('Normality == 1'))

    return featureDistrib, normalFeatures

In [24]:
distrib, numNormal = computeShapiro(frameVectorDf)

print(f"{numNormal}/64 features are normally distributed")
distrib.head(64)


8/64 features are normally distributed


,Normality
Gpd_1,0
Gpd_2,1
Gpd_3,0
Gpd_4,1
Gpd_5,0
...,...
Interperson_feature_5,0
Interperson_feature_6,0
Interperson_feature_7,0
Interperson_feature_8,0


In [25]:
print(distrib.query('Normality == 1'))

        Normality
Gpd_2           1
Gpd_4           1
Gpd_11          1
Gpd_12          1
Gpd_15          1
Gpd_22          1
Gpd_24          1
Gpd_44          1


In [26]:
distrib, numNormal = computeShapiro(frameVectorDf, 0.01)

print(f"{numNormal}/64 features are normally distributed")

10/64 features are normally distributed


Majority of features are non normal, thus we will proceed with feature ranking

In [27]:
# maps output to uniform distribution [0-1]
# random_state controls small random noise to handle tied values , setting it ensures reproducibility
#default n_quantiles = 100 so we change it
qt = QuantileTransformer(output_distribution='uniform', n_quantiles=626, random_state=42) 
frameMatrixScaled = qt.fit_transform(frameMatrix)

In [ ]:
##exporting frameMatrixScaled
import pickle 

with open('frameMatrixScaled.pkl', 'wb') as f:
    pickle.dump(frameMatrixScaled, f)

In [ ]:
##applying equal weighting to feature groups
w_gpd = 1/50
w_table = 1/4
w_inter = 1/9

gpdGrp = frameMatrixScaled[:, :50] * w_gpd
tableGrp = frameMatrixScaled[:, 50:55] * w_table
interpersonGrp = frameMatrixScaled[:, 55:] * w_inter
weightedFrames = np.concatenate([gpdGrp, tableGrp, interpersonGrp], axis=1)

In [ ]:
##exporting frameMatrixScaled

with open('weightedFrames.pkl', 'wb') as f:
    pickle.dump(weightedFrames, f)

In [ ]:
#applying AHP weighing
gpd_ahp = 0.122
table_ahp = 0.23
inter_ahp = 0.648

gpdGrp_ahp = frameMatrixScaled[:, :50] * gpd_ahp
tableGrp_ahp = frameMatrixScaled[:, 50:55] * table_ahp
interpersonGrp_ahp = frameMatrixScaled[:, 55:] * inter_ahp
ahp_weightedFrames = np.concatenate([gpdGrp_ahp, tableGrp_ahp, interpersonGrp_ahp], axis=1)

with open('ahpFrames.pkl', 'wb') as f:
    pickle.dump(ahp_weightedFrames, f)

In [ ]:
#------converting matrix into pandas data frame--

#creating cols and rows
dfCols = []
dfRows = []

interpersonIdx = 1
tableFeatIdx = 1
for i in range(1, np.shape(weightedFrames)[1] + 1):  
    if i <=50:
        dfCols.append(f"Gpd_{i}")

    elif i == 51:
        dfCols.append(f"Num_clinicians")
    
    elif i>=52 and i<=55:
        dfCols.append(f"Table_feature_{tableFeatIdx}")
        tableFeatIdx+=1
    
    else: 
        dfCols.append(f"Interperson_feature_{interpersonIdx}")
        interpersonIdx+=1

for i in range(1, np.shape(weightedFrames)[0]+1):
    dfRows.append(f"Frame_{i}")


scaledFrameDf = pd.DataFrame(
    weightedFrames,
    columns = dfCols,
    index = dfRows
)

scaledFrameDf.head()

In [ ]:
print(scaledFrameDf.to_string())

## Regroup per operation

In [ ]:
#group per operation again 
for day in annots:
    print(annots[day].keys())
    print(len(annots[day]))

In [ ]:
for day in annots:
    for operation in annots[day]:
        print(f"{day} {operation}: {len(annots[day][operation])}")

In [ ]:
OP_RANGES = [
    (0, 6),      # day3 op6: 7
    (7, 29),     # day3 op3: 23
    (30, 37),    # day3 op9: 8
    (38, 70),    # day3 op5: 33
    (71, 80),    # day3 op8: 10
    (81, 101),   # day3 op4: 21
    (102, 122),  # day3 op7: 21
    (123, 137),  # day3 op11: 15
    (138, 154),  # day3 op2: 17
    (155, 167),  # day3 op10: 13
    (168, 174),  # day3 op1: 7
    (175, 279),  # day2 op1: 105
    (280, 302),  # day2 op9: 23
    (303, 399),  # day2 op2: 97
    (400, 411),  # day2 op6: 12
    (412, 418),  # day2 op7: 7
    (419, 429),  # day2 op5: 11
    (430, 440),  # day2 op8: 11
    (441, 444),  # day2 op11: 4
    (445, 452),  # day2 op4: 8
    # (453, 454),  # day2 op3: 2
    # (455, 456),  # day2 op10: 2 <-- excluded due to length
    (457, 526),  # day4 op1: 70
    (527, 567),  # day4 op2: 41
    (568, 572),  # day4 op3: 5
    (573, 625),  # day1 op1: 53
]

In [ ]:
#converting pandas df to numpy arr
frames = scaledFrameDf.to_numpy()

In [ ]:
#grouping based on op_ranges
operationVectors = []
for r in OP_RANGES:
    op = frames[r[0]:r[1]+1,]
    operationVectors.append(op)

operationVectors[0]

In [ ]:
#exporting so we can work with operation list elsewhere

import pickle 

with open('operationList.pkl', 'wb') as f:
    pickle.dump(operationVectors, f)
    

## Preprocessing to check robustness of clustering to noise

In [ ]:
first90_operations =  [] #first 90% of frames in each operation
last90_operations = [] #last 90% of frames 

for operation in operationVectors:
    n = len(operation)
    cut = int(np.floor(0.9 * n))
    start = n - cut  # = index where last 90% begins

    first90_operations.append(operation[:cut])
    last90_operations.append(operation[start:])


In [ ]:
operations_over20 = [] #operations over 20 frames

for operation in operationVectors:
    if np.shape(operation)[0]>=20:
        operations_over20.append(operation)

In [ ]:
print(len(operations_over20))

In [ ]:
operation_chunks = [] 
for operation in operations_over20:
    for i in range(0, len(operation) - len(operation)%10, 10): #removes last incomplete portion of chunk
        operation_chunks.append(operation[i:i+10]) 


In [ ]:
print(len(operation_chunks))

In [ ]:
#exporting modified annotations
with open('first90percent.pkl', 'wb') as f:
    pickle.dump(first90_operations, f)

with open('last90percent.pkl', 'wb') as f:
    pickle.dump(last90_operations, f)

with open('operation_chunks.pkl', 'wb') as f:
    pickle.dump(operation_chunks, f)